# YOLO11 Ambulance Detection Training Pipeline
This notebook trains a YOLO11 model on the ambulance detection dataset with scaled images (384:-1 aspect ratio).

**Dataset Location:** `/mnt/NewVolume/IIIT-Internship/Week-5/dataset/`
- Training: 100 images with 100 label files
- Validation: 40 images with 40 label files
- Class: Ambulance (single class detection)

## 1. Import Required Libraries

In [2]:
!pip install ultralytics


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /mnt/NewVolume/Internship/Week-3/.venv/bin/python -m pip install --upgrade pip


In [3]:
import os
import sys
import torch
import yaml
from pathlib import Path
from ultralytics import YOLO

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Verify ultralytics installation
print(f"YOLO version: {YOLO.__module__}")

Using device: cpu
PyTorch version: 2.11.0+cu130
YOLO version: ultralytics.models.yolo.model


## 2. Load and Validate Dataset

In [4]:
# Dataset paths
dataset_root = Path("/mnt/NewVolume/IIIT-Internship/Week-5/dataset")
dataset_yaml = dataset_root / "dataset.yaml"

# Verify dataset structure
print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

# Check YAML file
if dataset_yaml.exists():
    with open(dataset_yaml) as f:
        dataset_config = yaml.safe_load(f)
    print(f"\n✓ dataset.yaml found")
    print(f"  Path: {dataset_config.get('path')}")
    print(f"  Classes: {dataset_config.get('nc')} ({dataset_config.get('names')})")
else:
    print(f"✗ dataset.yaml NOT found at {dataset_yaml}")

# Verify image and label directories
img_train = dataset_root / "images_scaled" / "train"
img_val = dataset_root / "images_scaled" / "val"
lbl_train = dataset_root / "labels" / "train"
lbl_val = dataset_root / "labels" / "val"

dirs_to_check = [
    ("Train Images", img_train),
    ("Val Images", img_val),
    ("Train Labels", lbl_train),
    ("Val Labels", lbl_val)
]

for name, path in dirs_to_check:
    if path.exists():
        file_count = len(list(path.glob("*")))
        print(f"✓ {name}: {file_count} files")
    else:
        print(f"✗ {name}: NOT found at {path}")

print(f"\n✓ Dataset ready for training!")
print(f"  Config file: {dataset_yaml}")

DATASET VALIDATION

✓ dataset.yaml found
  Path: /mnt/NewVolume/IIIT-Internship/Week-5/dataset
  Classes: 1 (['Ambulance'])
✓ Train Images: 100 files
✓ Val Images: 40 files
✓ Train Labels: 100 files
✓ Val Labels: 40 files

✓ Dataset ready for training!
  Config file: /mnt/NewVolume/IIIT-Internship/Week-5/dataset/dataset.yaml


## 3. Configure Training Parameters

In [5]:
# Training configuration
print("\n" + "=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

# Training parameters - optimized for single-class detection
EPOCHS = 100  # High epochs to monitor train/validation loss
BATCH_SIZE = 16
IMG_SIZE = 384  # Match our scaled image width
PATIENCE = 20  # Early stopping patience
LR = 0.001  # Learning rate

training_config = {
    "model": "yolov11n",  # YOLOv11 nano - lightweight
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "patience": PATIENCE,
    "lr0": LR,
    "device": device,
    "save": True,
    "verbose": True,
    "plots": True,  # Save training plots
}

print(f"\nTraining Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Output directory for results
output_dir = Path("/mnt/NewVolume/IIIT-Internship/Week-5/runs")
print(f"\nOutput directory: {output_dir}")
print(f"Results will be saved to: {output_dir}/detect/trainX/")


TRAINING CONFIGURATION

Training Configuration:
  model: yolov11n
  epochs: 100
  batch: 16
  imgsz: 384
  patience: 20
  lr0: 0.001
  device: cpu
  save: True
  verbose: True
  plots: True

Output directory: /mnt/NewVolume/IIIT-Internship/Week-5/runs
Results will be saved to: /mnt/NewVolume/IIIT-Internship/Week-5/runs/detect/trainX/


## 4. Train the Model

In [ ]:
print("\n" + "=" * 60)
print("STARTING MODEL TRAINING")
print("=" * 60)
print(f"\nTraining on {device.upper()}...")
print(f"Dataset YAML: {dataset_yaml}")
print(f"Model: YOLOv11 Nano")
print(f"Total Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print("\nMonitoring metrics:")
print("  - Train Loss (box, cls, dfl)")
print("  - Validation Loss (box, cls, dfl)")
print("  - mAP50, mAP50-95")
print("\n" + "-" * 60 + "\n")

# Initialize YOLO model
# Initialize YOLO model (correct Ultralytics YOLO11 weight name)
model = YOLO("yolo11n.pt")

# Train the model
results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=device,
    lr0=LR,
    save=True,
    verbose=True,
    plots=True,
    project="/mnt/NewVolume/IIIT-Internship/Week-5/runs",
    name="ambulance_detection"
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\n✓ Model training finished!")
print(f"Results saved to: {results.save_dir if hasattr(results, 'save_dir') else '/mnt/NewVolume/IIIT-Internship/Week-5/runs/detect/ambulance_detection'}")


STARTING MODEL TRAINING

Training on CPU...
Dataset YAML: /mnt/NewVolume/IIIT-Internship/Week-5/dataset/dataset.yaml
Model: YOLOv11 Nano
Total Epochs: 100
Batch Size: 16
Image Size: 384x384

Monitoring metrics:
  - Train Loss (box, cls, dfl)
  - Validation Loss (box, cls, dfl)
  - mAP50, mAP50-95

------------------------------------------------------------



FileNotFoundError: [Errno 2] No such file or directory: 'yolov11n.pt'

## 5. Evaluate Model Performance

In [ ]:
# Evaluate the model on validation set
print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

val_results = model.val(
    data=str(dataset_yaml),
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=device,
    verbose=True
)

print("\n✓ Validation complete!")
print(f"\nMetrics Summary:")
if hasattr(val_results, 'box') and hasattr(val_results, 'seg'):
    print(f"  Box mAP50: {val_results.box.map50 if hasattr(val_results.box, 'map50') else 'N/A'}")
    print(f"  Box mAP50-95: {val_results.box.map if hasattr(val_results.box, 'map') else 'N/A'}")
else:
    print("  Check results object for detailed metrics")

print("\nValidation results available in results object")

## 6. Save Trained Model

In [ ]:
print("\n" + "=" * 60)
print("MODEL SAVING")
print("=" * 60)

# Define paths for model weights
runs_dir = Path("/mnt/NewVolume/IIIT-Internship/Week-5/runs")
weights_dir = runs_dir / "detect" / "ambulance_detection" / "weights"

# Find the best model
if weights_dir.exists():
    best_model_path = weights_dir / "best.pt"
    last_model_path = weights_dir / "last.pt"
    
    if best_model_path.exists():
        print(f"\n✓ Best model found: {best_model_path}")
        print(f"  File size: {best_model_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    if last_model_path.exists():
        print(f"✓ Last model found: {last_model_path}")
        print(f"  File size: {last_model_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print(f"⚠ Weights directory not found at {weights_dir}")
    print("Training results are in the runs directory")

# Summary information
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\n✓ Training completed successfully!")
print(f"\nKey locations:")
print(f"  - Best weights: {weights_dir / 'best.pt' if weights_dir.exists() else 'Check runs directory'}")
print(f"  - Last weights: {weights_dir / 'last.pt' if weights_dir.exists() else 'Check runs directory'}")
print(f"  - Results directory: {runs_dir / 'detect' / 'ambulance_detection'}")
print(f"\nThe trained model is ready for inference (Task 4)!")
print(f"Model weights will be used to detect Ambulances on test images.")